# Parte Específica: EDA estrutural do BankChurners

Dataset: Credit Card Customers (BankChurners), arquivo `data/parte_especifica/BankChurners.csv`
(já sem as duas colunas `Naive_Bayes_Classifier_...`).

Pilar estrutural: dimensões, tipos das variáveis e qualidade dos dados.

In [ ]:
# [IA] Claude Code – importacao das bibliotecas e leitura do dataset
from pathlib import Path

import pandas as pd

CAMINHO_DADOS = Path("../../data/parte_especifica/BankChurners.csv")
df = pd.read_csv(CAMINHO_DADOS)
df.head()

## 1. Dimensões

In [ ]:
# [IA] Claude Code – quantidade de linhas e colunas
linhas, colunas = df.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

## 2. Tipos das variáveis

In [ ]:
# [IA] Claude Code – tabela de tipos: dtype do pandas, quantidade de valores distintos e exemplos
tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "qtd_valores_distintos": df.nunique(),
    "exemplos": [df[c].drop_duplicates().head(4).tolist() for c in df.columns],
})
tipos["grupo"] = ["numerica" if pd.api.types.is_numeric_dtype(df[c]) else "categorica" for c in df.columns]
tipos

In [ ]:
# [IA] Claude Code – resumo da quantidade de variaveis por grupo
tipos["grupo"].value_counts()

### Classificação das variáveis

Adotamos a classificação das variáveis apresentada no dicionário do dataset, considerando seus significados e seus tipos estatísticos.


## 3. Qualidade dos dados

In [ ]:
# [IA] Claude Code – valores ausentes (NaN) por coluna
ausentes = df.isna().sum()
print("Total de valores ausentes (NaN):", ausentes.sum())
ausentes[ausentes > 0]

In [ ]:
# [IA] Claude Code – linhas duplicadas e unicidade do identificador do cliente
print("Linhas duplicadas:", df.duplicated().sum())
print("CLIENTNUM repetidos:", df["CLIENTNUM"].duplicated().sum())

In [ ]:
# [IA] Claude Code – valores 'Unknown' nas variaveis categoricas (ausencia registrada como texto, nao como NaN)
colunas_categoricas = df.select_dtypes(exclude="number").columns

unknown = pd.DataFrame({
    "qtd_unknown": [(df[c] == "Unknown").sum() for c in colunas_categoricas],
}, index=colunas_categoricas)
unknown["pct_unknown"] = (unknown["qtd_unknown"] / len(df) * 100).round(2)
unknown.sort_values("qtd_unknown", ascending=False)

In [ ]:
# [IA] Claude Code – linhas com pelo menos um 'Unknown' e quantos 'Unknown' cada linha tem
qtd_unknown_por_linha = (df[colunas_categoricas] == "Unknown").sum(axis=1)
print("Linhas com pelo menos um Unknown:", (qtd_unknown_por_linha > 0).sum(),
      f"({(qtd_unknown_por_linha > 0).mean() * 100:.2f}%)")
qtd_unknown_por_linha.value_counts().sort_index().rename_axis("qtd_unknown_na_linha").rename("qtd_linhas")

In [ ]:
# [IA] Claude Code – categorias de cada variavel categorica (procura de grafias inconsistentes)
for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].value_counts().to_dict()}")

In [ ]:
# [IA] Claude Code – faixas das variaveis numericas (procura de valores impossiveis, ex.: negativos)
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM")

faixas = df[colunas_numericas].describe().T[["min", "max", "mean", "std"]].round(3)
faixas["qtd_negativos"] = (df[colunas_numericas] < 0).sum()
faixas

## 4. Variável alvo

In [ ]:
# [IA] Claude Code – contagem e proporcao das classes da variavel alvo
alvo = pd.DataFrame({
    "qtd": df["Attrition_Flag"].value_counts(),
    "pct": (df["Attrition_Flag"].value_counts(normalize=True) * 100).round(2),
})
alvo